# Key proof steps, checked

Small-$m$ numerical checks of the main steps (block law $z=(y,w)$, $z_0=y$, $n=m+1$; ones are encoded as $-1$).
Every cell ends with asserts; the notebook runs in seconds.

In [ ]:
import numpy as np, itertools as it
from math import log, log2
ln2 = log(2)

def parities(n):
    # Z: blocks in {+-1}^n (+1 first); Bs: subsets as 0/1 rows (row 0 = empty); X[z,B] = chi_B(z)
    Z = np.array(list(it.product([1, -1], repeat=n)))
    Bs = np.array(list(it.product([0, 1], repeat=n)))
    X = np.array([[np.prod(z[b == 1]) for b in Bs] for z in Z])
    return Z, Bs, X

def marginals_agree(p, n):
    P = p.reshape((2,) * n)
    return np.allclose(P.sum(0), P.sum(-1))        # (z_1..z_m) vs (z_0..z_{m-1})

def cond_bias(p, n):
    P = p.reshape((2,) * n)                        # g(w) = E[Y | W=w], and P(W=w)
    return ((P[0] - P[1]) / (P[0] + P[1])).ravel(), (P[0] + P[1]).ravel()

def block_from_g(g, m):
    # stationary block law of the chain with Pr(y|w) = (1+y g(w))/2
    W = list(it.product([1, -1], repeat=m)); idx = {w: i for i, w in enumerate(W)}
    T = np.zeros((2**m, 2**m))
    for w in W:
        for y in (1, -1): T[idx[w], idx[(y,) + w[:-1]]] += (1 + y * g[idx[w]]) / 2
    ev, V = np.linalg.eig(T.T); pi = np.real(V[:, np.argmax(np.real(ev))]); pi /= pi.sum()
    return np.array([pi[idx[w]] * (1 + y * g[idx[w]]) / 2 for y in (1, -1) for w in W])

def max_cycle_mean(g, m):
    # Karp: edge (w,y) -> (y,w_1..w_{m-1}) has weight log2(1+y g(w)); 1-h_inf = max cycle mean
    W = list(it.product([1, -1], repeat=m)); idx = {w: i for i, w in enumerate(W)}
    E = [(idx[w], idx[(y,) + w[:-1]], log2(1 + y * g[idx[w]])) for w in W for y in (1, -1)]
    n = len(W); D = np.zeros((n + 1, n))
    for k in range(1, n + 1):
        D[k] = -np.inf
        for u, v, c in E: D[k, v] = max(D[k, v], D[k - 1, u] + c)
    return max(min((D[n, v] - D[k, v]) / (n - k) for k in range(n)) for v in range(n))

## 1. Sharp prediction bound (Thm. 2): the extremal law for equal $\gamma_r\equiv\gamma$
$p_*(z)=2^{-n}[1+\gamma-\gamma\prod_j(1-z_j)]$ is a stationary block law with $b_B=(-1)^{|B|-1}\gamma$ and worst-context advantage $2^{m-1}\gamma/[1-(2^m-1)\gamma]$, attaining the upper bound.

In [ ]:
gam = 0.02
for m in range(1, 5):
    n = m + 1; Z, Bs, X = parities(n)
    assert np.allclose(X @ (-1.0) ** Bs.sum(1), np.prod(1 - Z, axis=1))      # sum_B (-1)^|B| chi_B(z) = prod(1-z_j)
    p = 2.0**-n * (1 + gam - gam * np.prod(1 - Z, axis=1))
    assert p.min() > 0 and np.isclose(p.sum(), 1) and marginals_agree(p, n)   # valid, stationary (Gamma<1)
    b = p @ X; s = Bs.sum(1)
    assert np.allclose(b[s > 0], (-1.0) ** (s[s > 0] - 1) * gam)              # b_B = (-1)^{|B|-1} gamma
    g, _ = cond_bias(p, n)
    assert np.isclose(np.abs(g).max() / 2, 2**(m - 1) * gam / (1 - (2**m - 1) * gam))
print("ok")

## 2. Auxiliary chain $q$ (proof of Thm. 4)
$q(z)=2^{-n}[1+\frac\sigma k\sum_{j<k}\chi_{B+j}(z)]$, $B\ni0$, $d=\max B$, $k=n-d$: a proper stationary law with all contexts of mass $\ge2^{-m}/k$, and $\mathbb E_q[y\,g(w)]=\sigma\widetilde b/k$ **exactly**, for any $g$.

In [ ]:
rng = np.random.default_rng(1)
for m in range(1, 5):
    n = m + 1; Z, Bs, X = parities(n)
    for _ in range(30):
        B = np.r_[1, rng.integers(0, 2, m)]; d = max(i for i in range(n) if B[i]); k = n - d
        sig = rng.choice([-1, 1])
        S = sum(np.prod(Z[:, [i + j for i in range(n) if B[i]]], axis=1) for j in range(k))
        q = 2.0**-n * (1 + sig / k * S)
        assert q.min() >= 0 and np.isclose(q.sum(), 1) and marginals_agree(q, n)
        assert q.reshape(2, -1).sum(0).min() >= 2.0**-m / k - 1e-12             # context mass
        g = rng.uniform(-1, 1, 2**m)                                            # any g
        chiA = np.prod(Z[:2**m, 1:][:, B[1:] == 1], axis=1)                     # chi_A(w), A = B minus {0}
        bt = (g * chiA).sum() / 2**m                                            # b-tilde: uniform context law
        assert np.isclose((q * Z[:, 0] * np.tile(g, 2)).sum(), sig * bt / k)
print("ok")

## 3. Lower bound on a random near-uniform source
Checks (i) $|b_{A\cup0}-\widetilde b|\le(2^m-1)\kappa\,\|g\|_\infty$ (total-variation step) and (ii) the heavy-cycle bound $\mu^*\ge(|\widetilde b|/k-\bar g^2)/\ln2$ from the proof, where $\mu^*=1-h_\infty$ (Karp).

In [ ]:
rng = np.random.default_rng(0)
for m in (1, 2, 3):
    n = m + 1; Z, Bs, X = parities(n)
    g = rng.uniform(-1, 1, 2**m) * 0.05
    p = block_from_g(g, m); b = p @ X
    kap = np.abs(b[1:]).max(); gb = np.abs(g).max(); mu = max_cycle_mean(g, m)
    for r in np.flatnonzero(Bs[:, 0] == 1):                                     # all B = {0} u A
        A = Bs[r, 1:]; k = n - max(i for i in range(n) if Bs[r, i])
        bt = (g * np.prod(Z[:2**m, 1:][:, A == 1], axis=1)).sum() / 2**m
        assert abs(b[r] - bt) <= (2**m - 1) * kap * gb + 1e-12                  # (i)
        assert mu >= (abs(bt) / k - gb**2) / ln2 - 1e-12                        # (ii)
print("ok")

## 4. Both constants are attained (Prop. 5)
$1-h_\infty=$ max cycle mean. Runs-of-ones: loop at the all-plus context, $\log_2(1+\theta)$, with $\kappa_m\approx(m+1)\theta$ (lower constant $1/((m+1)\ln2)$).
Trap: loop at $w_-$, $\log_2(1+x)$ with $x=2^m\gamma/[1+(2^m-1)\gamma]$, and $\kappa_m=\gamma$ (upper constant $2^m/\ln2$).

In [ ]:
for m in (1, 2, 3, 4):
    n = m + 1; Z, Bs, X = parities(n); W = list(it.product([1, -1], repeat=m))
    # runs-of-ones: g = theta (2^{l(w)+1} - 1), l = leading ones (-1)
    th = 1e-5; ell = lambda w: next((i for i, x in enumerate(w) if x == 1), m)
    g = th * np.array([2**(ell(w) + 1) - 1 for w in W]); p = block_from_g(g, m)
    kap = np.abs((p @ X)[1:]).max(); mu = max_cycle_mean(g, m)
    assert np.isclose(mu, log2(1 + th), rtol=1e-6) and np.isclose(mu / kap, 1 / ((m + 1) * ln2), rtol=1e-3)
    # all-ones trap
    ga = 1e-5; p = np.full(2**n, (1 - ga) / 2**n); p[-1] = (1 + (2**n - 1) * ga) / 2**n
    assert np.allclose(np.abs((p @ X)[1:]), ga)                                 # every |b_B| = gamma
    g, _ = cond_bias(p, n); mu = max_cycle_mean(g, m); x = 2**m * ga / (1 + (2**m - 1) * ga)
    assert np.isclose(mu, log2(1 + x), rtol=1e-6) and np.isclose(mu / ga, 2**m / ln2, rtol=1e-3)
print("ok")

## 5. Potential inequality (runs-of-ones, upper bound)
$\phi(w)=2^{\ell(w)+1}-2$ gives $s\,G(w)\le1+\phi(w)-\phi(w')$ for every step; equality except for $s=-1$ at $\ell=m$. Hence no path beats the all-plus loop.

In [ ]:
for m in range(1, 8):
    ell = lambda w: next((i for i, x in enumerate(w) if x == 1), m); G = lambda w: 2**(ell(w) + 1) - 1
    for w in it.product([1, -1], repeat=m):
        for s in (1, -1):
            lhs = s * G(w); rhs = 1 + (G(w) - 1) - (G((s,) + w[:-1]) - 1)
            assert lhs <= rhs and (lhs == rhs or (s == -1 and ell(w) == m))
print("ok")

## 6. Noisy parity (Thm. 8)
Exact $\mathrm{H}_{\min}$ by enumeration ($m=2$, $k=2$, lag $d_1=m$) and the length where the fair-coin rule starts to reject ($\varepsilon=0.01$).

In [ ]:
m, N, dl = 2, 10, 0.1; best = 0; outs = set()
for seed in it.product([0, 1], repeat=m):
    for xi in it.product([0, 1], repeat=N - m):
        s = list(seed)
        for t in range(m, N): s.append(s[t - m] ^ xi[t - m])
        outs.add(tuple(s)); pr = 2.0**-m * np.prod([0.5 - dl if v else 0.5 + dl for v in xi])
        best = max(best, pr)
assert len(outs) == 2**N                                                         # bijection
assert np.isclose(-log2(best), m + (N - m) * (1 - log2(1 + 2 * dl)))             # H_min formula

delta = (2**0.01 - 1) / 2; f = lambda N: 2 * delta * N / np.sqrt(4 * N * np.log(N))   # margin, k=2
lo, hi = 1e3, 1e9
for _ in range(100): mid = np.sqrt(lo * hi); lo, hi = (mid, hi) if f(mid) < 1 else (lo, mid)
print("margin = 1 at N ~ %.2e" % lo); assert 5e5 < lo < 5e6                        # 'order 10^6'

## 7. Why XOR lemmas lose (appendix)
$\|p-U\|_1\le(\sum_{B\ne\varnothing}b_B^2)^{1/2}$ and $\Delta\le\|p-U\|_1/(2\min_wP(w))$; with $\min_wP\ge2^{-m-1}$ this is $2^m\sqrt{2^{m+1}-1}\,\kappa$, a factor $\Theta(2^{m/2})$ above the sharp $2^{m-1}\kappa/(1-(2^m-1)\kappa)$.

In [ ]:
rng = np.random.default_rng(2)
for m in (1, 2, 3):
    n = m + 1; Z, Bs, X = parities(n)
    p = 2.0**-n * (1 + 0.1 * rng.uniform(-1, 1, 2**n)); p /= p.sum(); b = p @ X
    g, Pw = cond_bias(p, n)
    assert np.abs(p - 2.0**-n).sum() <= np.sqrt((b[1:]**2).sum()) + 1e-12       # Parseval
    assert np.abs(g).max() / 2 <= np.abs(p - 2.0**-n).sum() / (2 * Pw.min()) + 1e-12
for m in (2, 4, 6, 8):
    print(m, "XOR/sharp ratio = %.1f" % (2 * np.sqrt(2**(m + 1) - 1)), " ~ 2^{(m+3)/2} = %.1f" % 2**((m + 3) / 2))